<a href="https://akademie.datamics.com/kursliste/">![title](bg_datamics_top.png)</a><center><em>© Datamics</em></center><br><center><em>Besuche uns für mehr Informationen auf <a href='https://akademie.datamics.com/kursliste/'>www.akademie.datamics.com</a></em>

# Empfehlungssysteme

Willkommen zum Code-Notebook zur Lektion über Empfehlungssysteme (en. recommender systems). In dieser Lektion werden wir ein einfaches Empfehlungssystem mit Python und Pandas erstellen. Zusätzlich gibt es ein zweites Notebook *Fortgeschrittene Empfehlungssysteme mit Python*, indem wir ein komplexeres Empfehlungssystem für den gleichen Datensatz erstellen.

In diesem Notebook geht es daraum ein einfaches funktionierendes Empfehlungssystem zu erstellen, dass Items empfiehlt, die möglichst ähnlich zu einem bestimmten Item sind. In unserem Fall wird es sich um Filme handeln. Denkt aber bitte daran, dass dies kein ausgeklügeltes reales Empfehlungssystem ist; es sagt uns einfach nur, welche Filme sich in ihren Eigenschaften ähneln. Netflix und Co. beziehen das Nutzerverhalten und viel mehr Eigenschaften für ihre Empfehlungen.

Es gibt zu diesem Thema kein Projekt, aber jeder der möchte kann das fortgeschrittene Notebook *optional* selbst durcharbeiten.

Legen wir los!

## Libraries

In [ ]:
import numpy as np
import pandas as pd

## Die Daten laden

In [ ]:
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('U.data', sep='\t', names=column_names)

In [ ]:
df.head()

Verpassen wir den Filmen ihren Namen:

In [ ]:
movie_titles = pd.read_csv("Movie_Id_Titles")
movie_titles.head()

Jetzt können wir beide DataFrames zusammenführen:

In [ ]:
df = pd.merge(df,movie_titles,on='item_id')
df.head()

## Explorative Daten Analyse

Finden wir ein paar grundlegende Eingeschaften unserer Daten heraus. Wir können dazu die Daten im Allgemeinen und die am besten bewerteten Filme betrachten.

## Visualisierungs-Imports

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('white')
%matplotlib inline

Erstellen wir uns einen DataFrame für die Bewertungen (en. ratings). Wir betrachten die durchschnittliche Bewertung und die Anzahl an Bewertungen.

In [ ]:
df.groupby('title')['rating'].mean().sort_values(ascending=False).head()

In [ ]:
df.groupby('title')['rating'].count().sort_values(ascending=False).head()

In [ ]:
ratings = pd.DataFrame(df.groupby('title')['rating'].mean())
ratings.head()

Jetzt können wir die Spalte der Anzahl an Bewertungen (en. number of ratings) hinzufügen:

In [ ]:
ratings['num of ratings'] = pd.DataFrame(df.groupby('title')['rating'].count())
ratings.head()

Verschaffen wir uns mit einigen Histogrammen einen Überblick:

In [ ]:
plt.figure(figsize=(10,4))
ratings['num of ratings'].hist(bins=70)

In [ ]:
plt.figure(figsize=(10,4))
ratings['rating'].hist(bins=70)

In [ ]:
sns.jointplot(x='rating',y='num of ratings',data=ratings,alpha=0.5)

Okay! Jetzt wo wir ungefähr wissen, wie unsere Daten aussehen: lasst uns damit fortfahren eine einfache Empfehlung zu erstellen:

## Ähnliche Filme empfehlen

Dazu erstellen wir zuerst eine Matrix. Diese wird die User-IDs (Spalte: user_id) auf der einen Achse und die Filmtitel (Spalte: title) auf der anderen Achse tragen. Jede Zelle beinhaltet dann die Bewertung, die ein Nutzer einem Film gegeben hat. Dabei können wir gleich darauf achten, dass wir viele `NaN`-Werte erhalten werden.

In [ ]:
moviemat = df.pivot_table(index='user_id',columns='title',values='rating')
moviemat.head()

Am häufigsten bewertete Filme:

In [ ]:
ratings.sort_values('num of ratings',ascending=False).head(10)

Wählen wir nun 2 Filme: Star Wars, ein Sci-Fi Film, und Liar Liar, eine Komödie.

In [ ]:
ratings.head()

Wählen wir die Bewertungen der Nutzer für diese Filme:

In [ ]:
starwars_user_ratings = moviemat['Star Wars (1977)']
liarliar_user_ratings = moviemat['Liar Liar (1997)']
starwars_user_ratings.head()

Anschließend können wir `corrwith()` verwenden, um Korrelationen zwischen den beiden Pandas Series zu erhalten:

In [ ]:
similar_to_starwars = moviemat.corrwith(starwars_user_ratings)
similar_to_liarliar = moviemat.corrwith(liarliar_user_ratings)

Säubern wir das Ergebnis indem wir die NaN Werte entfernen. Außerdem können wir aus der Series einen DataFrame machen.

In [ ]:
corr_starwars = pd.DataFrame(similar_to_starwars,columns=['Correlation'])
corr_starwars.dropna(inplace=True)
corr_starwars.head()

Wenn wir diesen DataFrame nach den Korrelationen sortieren, dann sollten wir die ähnlichsten Filme erhalten.

Beachtet dabei, dass viele Nutzer Star Wars gesehen haben (der bekannteste Film) und andere Filme nur von einzelnen Nutzern gesehen wurden.

In [ ]:
corr_starwars.sort_values('Correlation',ascending=False).head(10)

Korrigieren wir unser Ergebnis deshalb noch um die Filme die weniger als 100 Bewertungen erhalten haben. Die Grenze muss nicht 100 sein, scheint aber nach Betrachtung des Histogramms eine sinnvolle Grenze zu sein.

In [ ]:
corr_starwars = corr_starwars.join(ratings['num of ratings'])
corr_starwars.head()

Jetzt sortieren wir die Werte erneut für Filme mit mehr als 100 Bewertungen. Dieses Ergebnis wird viel mehr Sinn ergeben:

In [ ]:
corr_starwars[corr_starwars['num of ratings']>100].sort_values('Correlation',ascending=False).head()

Das können wir auch für die Komödie Liar Liar machen:

In [ ]:
corr_liarliar = pd.DataFrame(similar_to_liarliar,columns=['Correlation'])
corr_liarliar.dropna(inplace=True)
corr_liarliar = corr_liarliar.join(ratings['num of ratings'])
corr_liarliar[corr_liarliar['num of ratings']>100].sort_values('Correlation',ascending=False).head()

# Gut gemacht!